# Assignment MMB

---

## Important Dates
- Homework release: Tue 22 Sep 2026
- **Homework due:** Mon 30 Nov 2026, 11:59 AM

---

This notebook is not graded, the markdown cells should help you create an error component model, step by step.

## Some rules
1. You are allowed to use any built-in Python library that comes with Anaconda. If you want to use an external library, you may do so, but must justify your choice.

2. Make sure you use the `data` folder provided in the repository in read-only mode. (Or alternatively, be sure you don’t change any of the files.) One should be able to reproduce your results using the original data files.

3. Be sure to provide a concise textual description of your thought process, the assumptions you made, the solution you implemented, and explanations for your answers.

4. Several questions ask to comment and analyze the results. Make sure to provide a clear interpretation of the results. A few lines of text should be enough to explain your reasoning and the implications of the results.

5. Please write all your comments in **English**, and use meaningful variable names in your code. 

6. Make sure to print results and/or dataframes that confirm you have properly addressed the task.


In [1]:
import biogeme.version as ver
print(ver.get_text())
# This assignment uses Biogeme 3.3.5.

biogeme 3.3.5 [2026-10-09]
Home page: http://biogeme.epfl.ch
Submit questions to https://groups.google.com/d/forum/biogeme
Michel Bierlaire, Transport and Mobility Laboratory, Ecole Polytechnique Fédérale de Lausanne (EPFL)



In [2]:
import numpy as np
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme.models import logit, loglogit, lognested, logcnl
from biogeme.nests import OneNestForNestedLogit, NestsForNestedLogit, OneNestForCrossNestedLogit, NestsForCrossNestedLogit
from biogeme.expressions import Beta, Variable, log, exp, RandomVariable, IntegrateNormal, Draws, MonteCarlo, Derive
from biogeme.results_processing import compile_estimation_results, get_pandas_estimated_parameters
from biogeme.tools import likelihood_ratio_test
from datetime import datetime as dt
print(f'Date: {dt.now().strftime("%Y-%m-%d %H:%M:%S")}')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Date: 2026-10-09 23:42:19


/home/pablo/MA3/MMoB/london-mode-choice/MMoB/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


# Model 0 

---

Start with a model specification that includes alternative specific constants, and cost and travel time of the different alternatives associated with generic parameters. Report both the specification (i.e., the utility functions) and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).  

Calculate the total public transport duration as the sum of `dur_pt_access`, `dur_pt_rail`, `dur_pt_bus`, and `dur_pt_int`. Similarly, calculate the total driving cost as the sum of `cost_driving_fuel` and `cost_driving_ccharge`.

1. Comment on the estimation output (sign and significance of all parameters, and goodness of fit indices).  

---

**0.1. – Preparing the data**  
- Load your assigned LPMC dataset.
- Compute `total_pt_duration = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int`.
- Compute `total_driving_cost = cost_driving_fuel + cost_driving_ccharge`.
- All individuals and all alternatives should be kept.

In [3]:
# Load the assigned data file (group 09). The data folder is only read, never modified.
df = pd.read_table('../data/lpmc09.dat')

# Create the two aggregate attributes requested in the question.
df['total_pt_duration'] = df['dur_pt_access'] + df['dur_pt_rail'] + df['dur_pt_bus'] + df['dur_pt_int']
df['total_driving_cost'] = df['cost_driving_fuel'] + df['cost_driving_ccharge']

# Create a Biogeme database from your dataframe.
database = db.Database('lpmc09', df)

print(f'Number of observations: {len(df)}')
display(df['travel_mode'].value_counts().sort_index().rename({1: 'walk', 2: 'cycle', 3: 'pt', 4: 'car'})
        .to_frame('count').assign(share=lambda x: x['count'] / x['count'].sum()))
display(df[['dur_walking', 'dur_cycling', 'total_pt_duration', 'dur_driving',
            'cost_transit', 'total_driving_cost', 'age', 'female']].describe().T)

Number of observations: 5000


,count,share
travel_mode,,
walk,815,0.1630
cycle,230,0.0460
pt,1773,0.3546
car,2182,0.4364


,count,mean,std,min,25%,50%,75%,max
dur_walking,5000.0,1.305683,1.248600,0.029444,0.379583,0.849444,1.856111,9.278333
dur_cycling,5000.0,0.418769,0.391812,0.006111,0.126667,0.277500,0.595000,2.660278
total_pt_duration,5000.0,0.504950,0.333123,0.016944,0.237431,0.429028,0.713611,2.363889
dur_driving,5000.0,0.322242,0.279509,0.009167,0.113889,0.225833,0.446736,1.855278
cost_transit,5000.0,1.909906,1.633134,0.000000,1.500000,1.500000,3.000000,13.490000
total_driving_cost,5000.0,2.339970,3.871111,0.030000,0.310000,0.710000,1.720000,16.600000
age,5000.0,45.099400,16.271179,19.000000,32.000000,43.000000,56.000000,94.000000
female,5000.0,0.050000,0.217967,0.000000,0.000000,0.000000,0.000000,1.000000


**0.2. – Model specification**  
- Define a model with:
  - Alternative-specific constants (ASCs),
  - A generic parameter for total cost,
  - A generic parameter for total travel time.
- Use the same parameter values across alternatives (generic parameters).


In [4]:
# Define a Variable object for every data column used in the utilities.
# Durations are in hours, costs in GBP.
travel_mode = Variable('travel_mode')
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
total_pt_duration = Variable('total_pt_duration')
dur_driving = Variable('dur_driving')
cost_transit = Variable('cost_transit')
total_driving_cost = Variable('total_driving_cost')
age = Variable('age')

In [5]:
# Define the choice variable, generic time and cost coefficients, and ASCs.
# Walking is the reference alternative: its ASC is not defined (fixed to 0) for identification.
asc_cycle = Beta('asc_cycle', 0, None, None, 0)
asc_pt = Beta('asc_pt', 0, None, None, 0)
asc_car = Beta('asc_car', 0, None, None, 0)

b_time = Beta('b_time', 0, None, None, 0)
b_cost = Beta('b_cost', 0, None, None, 0)

# All alternatives are available to all individuals.
availability = {1: 1, 2: 1, 3: 1, 4: 1}

In [6]:
# Write one utility for each alternative, then collect them in V_0.
# Walking and cycling have no monetary cost.
V_walk_0 = b_time * dur_walking
V_cycle_0 = asc_cycle + b_time * dur_cycling
V_pt_0 = asc_pt + b_time * total_pt_duration + b_cost * cost_transit
V_car_0 = asc_car + b_time * dur_driving + b_cost * total_driving_cost

V_0 = {
    1: V_walk_0,
    2: V_cycle_0,
    3: V_pt_0,
    4: V_car_0
}

**Specification of Model 0** (walk is the reference alternative):

$$
\begin{aligned}
V_{walk} &= \beta_{time}\, T_{walk} \\
V_{cycle} &= ASC_{cycle} + \beta_{time}\, T_{cycle} \\
V_{pt} &= ASC_{pt} + \beta_{time}\, T_{pt} + \beta_{cost}\, C_{pt} \\
V_{car} &= ASC_{car} + \beta_{time}\, T_{car} + \beta_{cost}\, C_{car}
\end{aligned}
$$

where $T$ are travel times in hours (with $T_{pt}$ = `total_pt_duration`), $C_{pt}$ = `cost_transit` and $C_{car}$ = `total_driving_cost` (in GBP).

**0.3. – Estimation of the parameters**  
- Estimate the model using Biogeme.
- Report:
  - Parameter estimates,
  - t-statistics or p-values,
  - Null and final log-likelihoods.

In [7]:
# Build the log-likelihood with loglogit, create a BIOGEME object, and estimate Model 0.
# Log-likelihood function:
# - util: dictionary with the utilities,
# - av: dictionary of availability (all alternatives available),
# - i: observed choice in the database.
logprob_0 = loglogit(util=V_0, av=availability, i=travel_mode)

# Create Biogeme object (HTML/YAML/iteration files are not needed here)
biogeme_0 = bio.BIOGEME(database, logprob_0, generate_html=False, generate_yaml=False, save_iterations=False)
biogeme_0.model_name = "model_0"
# Estimate the model
results_0 = biogeme_0.estimate()

In [8]:
# In Biogeme 3.3.5, the parameter tables are named: 
param_0 = get_pandas_estimated_parameters(estimation_results=results_0)
display(param_0['Estimated parameters'])

# Keep every estimated model in this dictionary for later comparisons.
all_results = {}
all_results['model_0'] = results_0

,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time,-4.973198,0.185989,-26.739260,0.0
1,asc_cycle,-3.274990,0.091985,-35.603436,0.0
2,asc_pt,-0.520112,0.054626,-9.521258,0.0
3,b_cost,-0.149278,0.011315,-13.193179,0.0
4,asc_car,-1.176688,0.078355,-15.017310,0.0


In [9]:
# In order to print general statistics such as the number of estimated parameters, the final log likelihood or the final gradient norm:

general_stats = results_0.get_general_statistics()
display(general_stats)

# or more synthetic
print(results_0.short_summary())

{'Number of estimated parameters': '5',
 'Sample size': '5000',
 'Excluded observations': '0',
 'Init log likelihood': '-6931.472',
 'Final log likelihood': '-4710.09',
 'Likelihood ratio test for the init. model': '4442.763',
 'Rho-square for the init. model': '0.32',
 'Rho-square-bar for the init. model': '0.32',
 'Akaike Information Criterion': '9430.181',
 'Bayesian Information Criterion': '9462.767',
 'Final gradient norm': '7.9073E-04',
 'Bootstrapping time': 'None'}

Results for model model_0
Nbr of parameters:		5
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4710.09
Akaike Information Criterion:	9430.181
Bayesian Information Criterion:	9462.767



**0.4. – Interpretation of the results**  
- Discuss the sign of each parameter and whether it aligns with expected behavior.
- Comment on the statistical significance of the coefficients.
- Evaluate the overall model fit.

<div class="alert alert-info">
  Your comments should be concise and to the point, focusing on the most relevant aspects of the model and its results.
</div>

**Comments on Model 0**

- **Signs.** Both $\beta_{time}$ and $\beta_{cost}$ are negative, as expected: longer and more expensive trips reduce the utility of a mode. Their ratio gives an implied value of time of about 33 GBP/hour. That is high, but it is a single average over very different modes and travellers.
- **Constants.** All three ASCs are negative relative to walking, so for the same time and cost the sample has an unexplained preference for walking. This is driven mostly by cycling ($ASC_{cycle} \approx -3.3$, only 4.6% of trips are by bike). The negative $ASC_{car}$ captures unobserved factors such as parking or not having a car or licence, none of which are in the model yet.
- **Significance.** Every parameter is highly significant ($|t| > 9$, p-values ≈ 0).
- **Fit.** The final log-likelihood is about −4710, against a null log-likelihood of $5000 \ln(1/4) \approx -6931$. This gives $\rho^2 \approx 0.32$, a reasonable fit for a first specification with 5 parameters.

# Model 1 

---

Using Model 0 as the base model, include alternative-specific parameters (i.e., different parameters per alternative) for one of the attributes of Model 0. Report both the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumption of defining alternative-specific parameters in this specific situation.  
2. Comment on the estimation output (as defined previously, including any changes from the previous model).  

3. Compare Model 0 and Model 1 with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**1.1. – Model modification**  
- Start from Model 0 and choose **one attribute** (e.g., cost or time).
- Replace the generic coefficient for that attribute with **alternative-specific parameters**.
  - For example: instead of one $\beta_{\text{time}}$, define $\beta_{\text{time, car}}$...


In [10]:
# Choose ONE attribute from Model 0 (here: travel time).
# Replace its generic coefficient with alternative-specific coefficients.
# Model 1 is nested in Model 0 (Model 0 is obtained with b_time_walk = b_time_cycle = b_time_pt = b_time_car).
b_time_walk = Beta('b_time_walk', 0, None, None, 0)
b_time_cycle = Beta('b_time_cycle', 0, None, None, 0)
b_time_pt = Beta('b_time_pt', 0, None, None, 0)
b_time_car = Beta('b_time_car', 0, None, None, 0)

V_walk_1 = b_time_walk * dur_walking
V_cycle_1 = asc_cycle + b_time_cycle * dur_cycling
V_pt_1 = asc_pt + b_time_pt * total_pt_duration + b_cost * cost_transit
V_car_1 = asc_car + b_time_car * dur_driving + b_cost * total_driving_cost

V_1 = {
    1: V_walk_1,
    2: V_cycle_1,
    3: V_pt_1,
    4: V_car_1
}

**1.2. – Behavioral interpretation**  
- Clearly state the behavioral assumption behind using different coefficients across alternatives.
  - Why might the sensitivity to that attribute differ by alternative?


**Specification of Model 1:** same as Model 0, but with alternative-specific time coefficients $\beta_{time,walk}$, $\beta_{time,cycle}$, $\beta_{time,pt}$ and $\beta_{time,car}$.

**Behavioural assumption.** One hour spent travelling does not cause the same disutility on every mode. Walking and cycling take physical effort and expose travellers to the weather. Driving involves congestion stress and the effort of driving. In public transport, travel time can partly be used for other activities (reading, phone), so it should be less onerous per hour. The marginal utility of time is therefore allowed to differ across modes, while the marginal utility of money (cost) stays generic, since a pound has the same value whatever mode it is spent on.

**1.3. – Estimation and output analysis**  
- Estimate the updated model.
- Report and interpret:
  - Parameter estimates and their signs,
  - Statistical significance (t-stats or p-values),
  - Null and final log-likelihoods,
  - Any noticeable changes compared to Model 0.


In [11]:
# Small helper used from now on: estimate a logit model from a dictionary of utilities.
def estimate_logit(utilities, model_name):
    logprob = loglogit(util=utilities, av=availability, i=travel_mode)
    biogeme_model = bio.BIOGEME(database, logprob, generate_html=False, generate_yaml=False, save_iterations=False)
    biogeme_model.model_name = model_name
    return biogeme_model.estimate()

results_1 = estimate_logit(V_1, 'model_1')
print(results_1.short_summary())
display(get_pandas_estimated_parameters(estimation_results=results_1)['Estimated parameters'])

Results for model model_1
Nbr of parameters:		8
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4416.83
Akaike Information Criterion:	8849.659
Bayesian Information Criterion:	8901.797



,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time_walk,-7.431550,0.398507,-18.648478,0.0
1,asc_cycle,-4.012184,0.174002,-23.058213,0.0
2,b_time_cycle,-5.033347,0.347192,-14.497318,0.0
3,asc_pt,-2.249014,0.137906,-16.308287,0.0
4,b_time_pt,-3.095652,0.231108,-13.394821,0.0
5,b_cost,-0.137408,0.012301,-11.170957,0.0
6,asc_car,-1.729702,0.136177,-12.701818,0.0
7,b_time_car,-5.679782,0.344189,-16.501911,0.0


In [12]:
# Add Model 1 before producing a comparison table.
all_results['model_1'] = results_1

comparison_table, _ = compile_estimation_results(all_results)
display(comparison_table)

,model_0,model_1
Number of estimated parameters,5,8
Sample size,5000,5000
Final log likelihood,-4710.09,-4416.83
Akaike Information Criterion,9430.181,8849.659
Bayesian Information Criterion,9462.767,8901.797
b_time (t-test),-4.97 (-26.7),
asc_cycle (t-test),-3.27 (-35.6),-4.01 (-23.1)
asc_pt (t-test),-0.52 (-9.52),-2.25 (-16.3)
b_cost (t-test),-0.149 (-13.2),-0.137 (-11.2)
asc_car (t-test),-1.18 (-15),-1.73 (-12.7)


**Comments on Model 1**

- **Signs and significance.** All four time coefficients are negative and highly significant. The cost coefficient barely changes (−0.149 → −0.137) and stays significant.
- **Ranking of time sensitivities.** In absolute value: walk (−7.4) > car (−5.7) > cycle (−5.0) > PT (−3.1). As expected, PT time is the least onerous and walking time the most. The implied values of time are about 22.5 GBP/h for PT and 41 GBP/h for car. Car users have a higher value of time, which is consistent with self-selection of higher-income travellers.
- **Constants.** The ASCs of PT and car become more negative. Since PT time is now valued less, a larger negative constant is needed to reproduce the observed PT share.
- **Fit.** The log-likelihood improves from −4710.1 to −4416.8 with 3 extra parameters ($\rho^2$ rises from about 0.32 to about 0.36).

**1.4. – Model comparison and selection**  
- Perform a likelihood ratio test (or another appropriate test) to compare Model 0 and Model 1.
- Clearly state:
  - The null hypothesis,
  - The degrees of freedom used,
  - The value of the test statistic and p-value,
  - The conclusion: which model is preferred ($\text{Model}_\text{pref}$)?

In [13]:
# Model 0 is the restricted model and Model 1 is the unrestricted model.
# H0: b_time_walk = b_time_cycle = b_time_pt = b_time_car (3 linear restrictions).
# The models are nested, so the likelihood ratio test applies:
# -2 (LL_0 - LL_1) ~ chi2 with (K_1 - K_0) degrees of freedom under H0.

In [14]:
# Extract log-likelihoods and number of estimated parameters
logL_0 = all_results['model_0'].final_loglikelihood
nparam_0 = all_results['model_0'].number_of_parameters # or number_of_free_parameters


logL_1 = all_results['model_1'].final_loglikelihood
nparam_1 = all_results['model_1'].number_of_parameters

# Run likelihood ratio test: model_0 is restricted, model_1 is unrestricted
lr_result = likelihood_ratio_test(
    model1=(logL_0, nparam_0),
    model2=(logL_1, nparam_1),
    significance_level=0.05
)
display(lr_result)

# Display results
print(f"Null hypothesis: b_time_walk = b_time_cycle = b_time_pt = b_time_car (Model 0 is sufficient)")
print(f"Degrees of freedom: {nparam_1 - nparam_0}")
print(f"Log-likelihood Model 0: {logL_0:.2f}")
print(f"Log-likelihood Model 1: {logL_1:.2f}")
print(f"Likelihood Ratio Statistic: {lr_result.statistic:.2f}")
print(f"Chi-squared Threshold (95%): {lr_result.threshold:.2f}")
print(f"{lr_result.message}")

LRTuple(message='H0 can be rejected at level 5.0%', statistic=586.5217158347314, threshold=np.float64(7.814727903251179))

Null hypothesis: b_time_walk = b_time_cycle = b_time_pt = b_time_car (Model 0 is sufficient)
Degrees of freedom: 3
Log-likelihood Model 0: -4710.09
Log-likelihood Model 1: -4416.83
Likelihood Ratio Statistic: 586.52
Chi-squared Threshold (95%): 7.81
H0 can be rejected at level 5.0%


**Likelihood ratio test.** The test is valid because Model 0 is a restricted version of Model 1: it imposes 3 linear equality restrictions on the time coefficients.

- $H_0$: $\beta_{time,walk} = \beta_{time,cycle} = \beta_{time,pt} = \beta_{time,car}$.
- Statistic: $-2(\mathcal{L}_0 - \mathcal{L}_1) = -2(-4710.1 + 4416.8) \approx 586.5$, compared with $\chi^2_{0.95,3} = 7.81$.
- $H_0$ is clearly rejected, so the time sensitivities differ across modes.

$\Rightarrow$ $\text{Model}_\text{pref}$ = **Model 1**.

# Model 2 

---

Using $\text{Model}_\text{pref}$ as the base model, choose a socioeconomic characteristic and define two model specifications. In the first specification, interact the socioeconomic characteristic with an alternative-specific constant. In the second specification, interact it with an alternative attribute. Report both the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumptions in this specific situation and how this specification explains behavior differently from $\text{Model}_\text{pref}$.  
2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$ and **Model 2** (both specifications) with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**2.0. – Model extension: interaction with socioeconomic variable**  
- Choose a socioeconomic characteristic from the dataset (e.g., age, gender).
- Create **two different model specifications**:
  - One where the characteristic interacts with an **alternative-specific constant**.
  - One where it interacts with an **attribute** (e.g., cost or time).


In [15]:
# Socioeconomic characteristic: age.
# Gender is not used because only 5% of the trips in our sample are made by women.
# Age is centred at 45 (close to the sample mean) and measured in decades, so the ASCs and time
# coefficients of Model_pref keep their interpretation for a 45-year-old traveller.
age_centred = (age - 45) / 10

# --- Specification 2a: age interacting with the alternative-specific constants ---
b_age_cycle = Beta('b_age_cycle', 0, None, None, 0)
b_age_pt = Beta('b_age_pt', 0, None, None, 0)
b_age_car = Beta('b_age_car', 0, None, None, 0)

V_2a = {
    1: V_walk_1,
    2: V_cycle_1 + b_age_cycle * age_centred,
    3: V_pt_1 + b_age_pt * age_centred,
    4: V_car_1 + b_age_car * age_centred
}

# --- Specification 2b: age interacting with the travel time attribute ---
# A generic shift of every time coefficient: beta_time,i(age) = beta_time,i + b_time_age * (age - 45) / 10
b_time_age = Beta('b_time_age', 0, None, None, 0)

V_2b = {
    1: V_walk_1 + b_time_age * age_centred * dur_walking,
    2: V_cycle_1 + b_time_age * age_centred * dur_cycling,
    3: V_pt_1 + b_time_age * age_centred * total_pt_duration,
    4: V_car_1 + b_time_age * age_centred * dur_driving
}

results_2a = estimate_logit(V_2a, 'model_2a_age_asc')
results_2b = estimate_logit(V_2b, 'model_2b_age_time')

for results in [results_2a, results_2b]:
    print(results.short_summary())
    display(get_pandas_estimated_parameters(estimation_results=results)['Estimated parameters'])

Results for model model_2a_age_asc
Nbr of parameters:		11
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4370.098
Akaike Information Criterion:	8762.196
Bayesian Information Criterion:	8833.885



,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time_walk,-7.561150,0.408701,-18.500457,0.000000
1,asc_cycle,-4.060615,0.178408,-22.760324,0.000000
2,b_time_cycle,-5.119460,0.352788,-14.511419,0.000000
3,b_age_cycle,0.025945,0.050397,0.514804,0.606690
4,asc_pt,-2.311557,0.141278,-16.361723,0.000000
5,b_time_pt,-3.049091,0.228809,-13.325911,0.000000
6,b_cost,-0.142597,0.012865,-11.084480,0.000000
7,b_age_pt,0.130966,0.032454,4.035479,0.000054
8,asc_car,-1.851607,0.141448,-13.090402,0.000000
9,b_time_car,-5.452151,0.343682,-15.863957,0.000000


Results for model model_2b_age_time
Nbr of parameters:		9
Sample size:			5000
Excluded data:			0
Final log likelihood:		-4363.731
Akaike Information Criterion:	8745.462
Bayesian Information Criterion:	8804.116



,Name,Value,Robust std err.,Robust t-stat.,Robust p-value
0,b_time_walk,-7.716564,0.392160,-19.677091,0.0
1,b_time_age,-0.755824,0.079963,-9.452176,0.0
2,asc_cycle,-4.088954,0.171542,-23.836424,0.0
3,b_time_cycle,-5.212265,0.338528,-15.396868,0.0
4,asc_pt,-2.281197,0.134879,-16.912921,0.0
5,b_time_pt,-3.261428,0.229962,-14.182470,0.0
6,b_cost,-0.150466,0.013274,-11.335185,0.0
7,asc_car,-1.836893,0.134550,-13.652079,0.0
8,b_time_car,-5.710506,0.338947,-16.847766,0.0


**2.1. – Behavioral interpretation**  
- Explain what each interaction is capturing in terms of behavior.
- How does this differ from $\text{Model}_\text{pref}$?
- What new insight does each interaction provide?

**Specifications** (with $a_n = (\text{age}_n - 45)/10$, starting from Model 1):

- **Model 2a (age × ASC):** $V_i = V_i^{(1)} + \beta_{age,i}\, a_n$ for $i \in \{cycle, pt, car\}$ (walk stays the reference).
- **Model 2b (age × time):** $V_i = V_i^{(1)} + \beta_{time,age}\, a_n\, T_i$ for all $i$, i.e. $\beta_{time,i}(a_n) = \beta_{time,i} + \beta_{time,age}\, a_n$.

**Behavioural assumptions.**

- **2a.** Age shifts the intrinsic preference for each mode independently of its level of service. Older travellers may value the comfort of car or PT, may be less willing to make physical effort, and are more likely to own a car. In Model 1 every traveller had the same constants. Here the unobserved "taste" for a mode varies systematically with age.
- **2b.** Age changes the sensitivity to travel time. A negative $\beta_{time,age}$ means older travellers find each extra minute of travel more burdensome (fatigue, discomfort), especially for long active trips. In Model 1 the time sensitivity was the same for everybody. Here it is heterogeneous across the population, which also makes the value of time depend on age.

An interaction of age with cost (income effect) was also tried but was not significant (t ≈ 0.97), so it is not reported.

**2.2. – Estimation and result interpretation**  
- Estimate both specifications of Model 2.
- For each:
  - Report parameter estimates, t-stats or p-values.
  - Compare null and final log-likelihoods.
  - Comment on changes in signs, magnitudes, or statistical significance compared to $\text{Model}_\text{pref}$.


In [16]:
# Store your estimated Model 2 results before comparing models.
all_results['model_2a_age_asc'] = results_2a
all_results['model_2b_age_time'] = results_2b

comparison_table, _ = compile_estimation_results(all_results)
display(comparison_table)

,model_0,model_1,model_2a_age_asc,model_2b_age_time
Number of estimated parameters,5,8,11,9
Sample size,5000,5000,5000,5000
Final log likelihood,-4710.09,-4416.83,-4370.098,-4363.731
Akaike Information Criterion,9430.181,8849.659,8762.196,8745.462
Bayesian Information Criterion,9462.767,8901.797,8833.885,8804.116
b_time (t-test),-4.97 (-26.7),,,
asc_cycle (t-test),-3.27 (-35.6),-4.01 (-23.1),-4.06 (-22.8),-4.09 (-23.8)
asc_pt (t-test),-0.52 (-9.52),-2.25 (-16.3),-2.31 (-16.4),-2.28 (-16.9)
b_cost (t-test),-0.149 (-13.2),-0.137 (-11.2),-0.143 (-11.1),-0.15 (-11.3)
asc_car (t-test),-1.18 (-15),-1.73 (-12.7),-1.85 (-13.1),-1.84 (-13.7)


**Comments on Model 2**

- **Model 2a.** $\beta_{age,car} \approx 0.25$ and $\beta_{age,pt} \approx 0.13$ are positive and significant. Every 10 years of age increase the preference for car (and, less strongly, PT) relative to walking. $\beta_{age,cycle}$ is not significant (t ≈ 0.5), which is not surprising with only 230 cycling trips. The other parameters keep their sign and significance and change only slightly in magnitude. The log-likelihood goes from −4416.8 to −4370.1 with 3 extra parameters.
- **Model 2b.** $\beta_{time,age} \approx -0.76$ (t ≈ −9.5): the disutility of time grows with age. For example, the walking time coefficient goes from about −7.7 at age 45 to about −9.2 at age 65. The other parameters remain stable and significant. The log-likelihood goes from −4416.8 to −4363.7 with only 1 extra parameter.
- **Both specifications** are behaviourally plausible and improve on $\text{Model}_\text{pref}$. 2b achieves the larger improvement with fewer parameters.

**2.3. – Model comparisons and updated Modelpref**  
- Use appropriate statistical tests to compare $\text{Model}_\text{pref}$ with each specification of Model 2.
- For each test:
  - Justify your choice of test,
  - State the null hypothesis,
  - Report the result of the test,
  - Draw a conclusion and update the preferred model if applicable.

In [17]:
# Model 1 (Model_pref) is nested in both 2a and 2b -> likelihood ratio tests.
# 2a and 2b are NOT nested in each other -> compare them with the adjusted rho-bar squared
# and the Horowitz non-nested test.
from scipy.stats import norm

logL_pref, nparam_pref = results_1.final_loglikelihood, results_1.number_of_parameters
for name, results in [('2a (age x ASC)', results_2a), ('2b (age x time)', results_2b)]:
    lr = likelihood_ratio_test(
        model1=(logL_pref, nparam_pref),
        model2=(results.final_loglikelihood, results.number_of_parameters),
        significance_level=0.05
    )
    print(f'Model 1 vs Model {name}: H0 = the interaction parameters are all zero')
    print(f'  df = {results.number_of_parameters - nparam_pref}, LR = {lr.statistic:.2f}, '
          f'threshold = {lr.threshold:.2f} -> {lr.message}\n')

# Non-nested comparison between 2a and 2b
L0 = len(df) * np.log(1 / 4)  # null log-likelihood (equal probabilities, 4 alternatives)
def rho_bar_square(results):
    return 1 - (results.final_loglikelihood - results.number_of_parameters) / L0

rho_bar_2a = rho_bar_square(results_2a)
rho_bar_2b = rho_bar_square(results_2b)
print(f'Adjusted rho-bar squared: 2a = {rho_bar_2a:.4f}, 2b = {rho_bar_2b:.4f}')

# Horowitz test: upper bound on the probability that the model with the lower rho-bar squared
# is the true one, P <= Phi(-sqrt(-2 (rho_bar_H - rho_bar_L) L(0) + (K_H - K_L)))
horowitz_p = norm.cdf(-np.sqrt(-2 * (rho_bar_2b - rho_bar_2a) * L0
                               + (results_2b.number_of_parameters - results_2a.number_of_parameters)))
print(f'Horowitz test: probability that 2a is the true model is at most {horowitz_p:.4f}')

Model 1 vs Model 2a (age x ASC): H0 = the interaction parameters are all zero
  df = 3, LR = 93.46, threshold = 7.81 -> H0 can be rejected at level 5.0%

Model 1 vs Model 2b (age x time): H0 = the interaction parameters are all zero
  df = 1, LR = 106.20, threshold = 3.84 -> H0 can be rejected at level 5.0%

Adjusted rho-bar squared: 2a = 0.3679, 2b = 0.3691
Horowitz test: probability that 2a is the true model is at most 0.0001


**Tests and updated $\text{Model}_\text{pref}$**

- **Model 1 vs 2a (LR test, 3 df).** $H_0$: $\beta_{age,cycle} = \beta_{age,pt} = \beta_{age,car} = 0$. LR ≈ 93.5 > 7.81, so $H_0$ is rejected.
- **Model 1 vs 2b (LR test, 1 df).** $H_0$: $\beta_{time,age} = 0$. LR ≈ 106.2 > 3.84, so $H_0$ is rejected.
- **2a vs 2b.** These models are not nested, so the LR test cannot be used. 2b has the higher adjusted $\bar{\rho}^2$, and the Horowitz test bounds the probability that 2a is the true model at well below 1%.

$\Rightarrow$ $\text{Model}_\text{pref}$ = **Model 2b** (alternative-specific time coefficients with an age × time interaction).

# Model 3 

---

Using $\text{Model}_\text{pref}$ as the base model, include an appropriate non-linear transformation on the time variable. Report both the specification and the estimation results (as defined previously).

1. State the behavioral assumption of the non-linear specification defined in this situation and how this specification explains behavior differently from $\text{Model}_\text{pref}$.  

2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$ and **Model 3** with an appropriate statistical test. Justify your choice of test. State the null hypothesis and the result of the test. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**3.0. – Model extension: non-linear transformation**  
- Apply a non-linear transformation to the time variable in a behaviorally justified way.
  - Examples include:
    - Logarithmic: $\log(\text{time})$
    - Piecewise linear functions
    - Squared or thresholded terms

**3.1. – Behavioral interpretation**  
- Explain the motivation behind using a non-linear transformation.
- What behavioral mechanism does this better represent?

**3.2. – Estimation and output analysis**  
- Estimate the model using the transformed specification.
- Report:
  - Parameter estimates,
  - Statistical significance,
  - Null and final log-likelihoods.
- Compare qualitatively with $\text{Model}_\text{pref}$.

In [18]:
# Store your estimated Model 3 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**3.3. – Model comparison and updated Modelpref**  
- Use appropriate statistical tests to compare $\text{Model}_\text{pref}$ with each specification of Model 3.
- For each test:
  - Justify your choice of test,
  - State the null hypothesis,
  - Report the result of the test,
  - Draw a conclusion and update the preferred model if applicable.

# Model 4 

---

Using $\text{Model}_\text{pref}$ as the base model, propose and test both a **nested** and a **cross-nested** structure. Report both structures using graphs, together with the specification and the estimation results (parameter values, t-tests or p-values, null and final log likelihoods).

1. State the behavioral assumptions of the proposed nesting structures in this specific situation.  

2. Comment on the estimation output (as defined previously).  

3. Compare $\text{Model}_\text{pref}$, the nested model, and the cross-nested model using appropriate statistical tests. Justify your choice of test(s). State the null hypotheses and the result of the tests. Denote the preferred model as $\text{Model}_\text{pref}$.  

---

**4.0.a – Model extension: nested logit structure**  
- Define a **nested logit model** by grouping alternatives that share unobserved factors (e.g., private vs. public modes).
- Provide a **graphical representation** of the nesting structure.
- Specify the utility functions and nest scale parameters ($\mu$) accordingly.

**4.0.b – Model extension: cross-nested logit structure**  
- Define a **cross-nested logit model**, allowing alternatives to belong to multiple nests (e.g., car might belong partially to both private and fast modes).
- Represent the cross-nesting structure with a **graph**.
- Specify the allocation parameters and scale parameters required by the model.


**4.1. – Behavioral interpretation**  
- Explain the behavioral rationale for the proposed nestings.
  - Why are certain alternatives likely to share unobserved factors?
- How does this structure improve over $\text{Model}_\text{pref}$?

**4.2. – Estimation and result interpretation**  
- Estimate both models:
  - Report parameter values, nest parameters (scale or allocation), and statistical significance.
  - Include null and final log-likelihoods.
- Discuss the estimated degree of correlation within and across nests.

In [19]:
# Store each estimated Model 4 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**4.3. – Model comparisons and updated Modelpref**  
- Use appropriate statistical tests to:
  - Compare $\text{Model}_\text{pref}$ with the nested model,
  - Compare the nested model with the cross-nested model,
- For each comparison:
  - State the null hypothesis,
  - Report the test statistic and degrees of freedom,
  - Justify which model to retain as the updated $\text{Model}_\text{pref}$.

# Model 5 

---
Using the best model from Model 3 as a starting point, propose a **mixture model** that captures correlations across alternatives similar to the one modeled in the best nested structure of Model 4. Relax the independence of the error terms across alternatives. Estimate the model and report its structure and results.

1. State the behavioral assumption behind your proposed mixture specification. What type of correlation does it capture ?  

2. Comment on the estimation output (as defined previously).  

3. Compare the estimates of this model with the estimates from the best performing model from Model 4.



**5.0. – Model extension: relaxation of independence through mixtures**  
- Start from the best model obtained in Model 3.
- Propose a **mixture model** to relax the independence of error terms across alternatives.
- Your goal is to reproduce the best nested substitution patterns from Model 4, with a mixture model.
- Clearly specify the mixture structure and the utility function for each case.

**5.1. – Behavioral interpretation**  
- Explain the behavioral assumption underlying the use of a mixture.
  - What types of substitution patterns or unobserved heterogeneity does this model capture?
  - How does this differ from the assumptions in Model 4?

**5.2. – Estimation and output analysis**  
- Estimate the model using the transformed specification.
- Report:
  - Parameter estimates,
  - Statistical significance,
  - Null and final log-likelihoods.

In [20]:
# Store your estimated Model 5 result before comparing models.
# all_results['your_model_name'] = results_your_model_name

# comparison_table, _ = compile_estimation_results(all_results)
# display(comparison_table)

**5.3. - Compare the estimates with those from the Model 4**
- Compare with an appropriate method the estimates of this model with the estimates from Model 4.

<div class="alert alert-info">

 Denote this mixture model as **$\text{Model}_\text{pref}$**, and use it as the reference model for all remaining questions (Market Shares and Forecasting).  

</div>

# Market Shares 

---

Assume that stratified random sampling was used to produce your sample. We consider the following strata:

- S1: females aged 42 years or younger  
- S2: females aged 43 years or older  
- S3: males aged 42 years or younger  
- S4: males aged 43 years or older

The table below gives the size of each category in the full population:

|               | Age < 43     | Age ≥ 43    |
|---------------|--------------|-------------|
| **Male**      | 2,926,408    | 1,379,198   |
| **Female**    | 2,841,376    | 1,519,948   |

(Source: ONS, 2015)

---

In [21]:
# 6.1: Define the four strata, count the sample size in each one, and calculate a weight from the population and sample shares.
# Attach the appropriate stratum weight to every observation.
# 6.2: Simulate individual probabilities, multiply by the weights, then aggregate and normalize by the total weight.
# 6.3: For observed shares, group observations by observed choice, sum the weights, and normalize by the total weight.

**6.1. – Compute stratum weights**  
- Based on your sample, compute the number of individuals in each stratum (S1–S4).
- Calculate the **weight** of each stratum.
$$
\omega_g = \frac{N_g}{N} \frac{S}{S_g}=\frac{\text{share of persons in
    segment $g$ in population}}{\text{share of persons in segment $g$ in sample}}
$$
- Report the size and weight of each stratum.  



**6.2. – Predict weighted market shares using Modelpref**  

- Use $\text{Model}_\text{pref}$ and the stratum weights to compute the **predicted market share** of each transport mode.
- Also compute **confidence intervals** for the predicted shares.
- Comment: Do the results match your expectations?  

<div class="alert alert-warning">

  <strong>Warning:</strong>\
   Depending on your machine, it might take too long to use bootstrapping for confidence intervals.
  If it's the case, you can use ```use_bootstrap=False```, in ```get_betas_for_sensitivity_analysis``` to disable bootstrapping.\
  The disadvantage is that it relies on the assumption that the estimates follow a normal distribution.
</div>

**6.3. – Compare with weighted observed shares**  

- Compute the actual weighted market shares in your sample (using the actual observed choices and stratum weights).
- Compare these to the model-predicted shares.  

# Forecasting 

---

Consider the following scenarios:

1. An additional charge of £1.5 for car users  
2. A 20% decrease in public transport cost

---

In [22]:
# 7.1: Start from a baseline simulation; change only the attribute affected by each scenario, then simulate and aggregate weighted shares again.
# 7.2: Compare the weighted car shares from the baseline and scenarios.
# 7.3: For each scenario, combine the PT probability, scenario fare, and observation weight before aggregating revenue.
# 7.4: Obtain marginal utilities of time and cost, form their ratio with consistent units, then calculate weighted averages.
# 7.5: Calculate individual probabilities and derivatives, form direct/cross elasticities, then aggregate them with the appropriate weights.

**7.1. – Simulate market shares under policy scenarios**  

- Use $\text{Model}_\text{pref}$ to predict the market shares for each of the two policy scenarios:
  - Scenario 1: increase the cost of car by £1.5
  - Scenario 2: decrease the cost of public transport by 20%
- Compare the predicted market shares to the original (baseline) shares.
- Comment: Do the results match your expectations?  


**7.2. – Policy evaluation: car reduction**  

- Which scenario is more effective at **reducing the market share of car**?
- Justify your answer based on the model results.  

**7.3. – Revenue analysis**  

- Compute the **total revenue** from public transport under each scenario.
- Identify which scenario yields the highest revenue.
- Compare with the revenue in the original situation.
- Provide an explanation for the result.  

**7.4. – Value of time**  

- Compute the **average value of time** (VOT) for car and for public transport.
- Report the results and comment on them.  

**7.5. – Elasticity analysis**  

- Compute the following **aggregate elasticities**:
  - **Direct elasticities**:
    - Car cost $\rightarrow$ car choice
    - PT cost $\rightarrow$ PT choice
  - **Cross elasticities**:
    - Car cost $\rightarrow$ PT, Cycle and Walk
    - PT cost $\rightarrow$ Car, Cycle and Walk
- A total of 8 values must be reported.
- Comment on the signs and magnitudes of the elasticities.  